# Direction 2 · Phase C · Step 7a: Waterbirds `ViT-B/16` ERM Training & Mechanism Diagnostic (Kaggle GPU)

**Settings:** Enable **GPU T4 x1** and **Internet ON**.

**Safe to run in parallel with Step 6b** (zero dependency on Step 6b & **zero Waterbirds test-split access**):
- Clones `spurious_correlation_mechanisms` from GitHub.
- Auto-downloads Waterbirds (~490 MB) and Caltech `CUB_200_2011` masks (~39 MB).
- Trains 3 standard-ERM `ViT-B/16` backbones (`seeds 0, 1, 2`, 20 epochs, no augmentation, `split == 0`) and persists checkpoints in `/kaggle/working/vit_b16_backbones/` for Step 7b.
- Runs validation-only (`split == 1`, cleaned `"relabel"` labels) 5-fold controls (`erm_raw`, `dfr_l2`, `oracle_fg_token_pool`, `ln_dfr_late`, `ln_dfr_all`), 5-view counterfactuals, block-wise foreground-token drift (`embed`..`block_11` + `[CLS]`), and spatial token patching (`orig <-> neutral_gray`).
- Saves compact diagnostic bundle to `/kaggle/working/rep_repair_step7a_vit_diag.zip`.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch

REPO_URL = 'https://github.com/chautanphat/waterbirds-shortcut-mitigation.git'
BRANCH = 'spurious_correlation_mechanisms'
CODE_DIR = Path('/kaggle/working/repo')
WORK_DIR = Path('/kaggle/working/step7a_assets')
CKPT_DIR = Path('/kaggle/working/vit_b16_backbones')
OUT_DIR = Path('/kaggle/working/rep_repair_step7a_vit_diag')

print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CODE_DIR)], check=True)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Pre-flight unit tests.
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_rep_repair_step7a_vit.py', '-v'], check=True)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cmd = [
    sys.executable, '-B', '-m', 'scripts.analysis.rep_repair_step7a_vit_diag',
    '--work-dir', str(WORK_DIR),
    '--ckpt-dir', str(CKPT_DIR),
    '--output', str(OUT_DIR),
    '--device', device,
    '--workers', '2',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((OUT_DIR / 'README.md').read_text(encoding='utf-8')))
display(Image(filename=str(OUT_DIR / 'step7a_vit_mechanism.png')))

print('Saved ViT-B/16 checkpoints (retained in /kaggle/working/vit_b16_backbones for Step 7b):')
for p in sorted(CKPT_DIR.glob('*.pt')):
    print(f'  {p.name}: {p.stat().st_size / 1e6:.1f} MB')

zip_path = Path('/kaggle/working/rep_repair_step7a_vit_diag.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.rglob('*')):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT_DIR))
print(f'Saved diagnostic bundle: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')